#### Import packages

In [ ]:
import pandas as pd
import numpy as np
import pickle
import datetime as dt
import seaborn as sns
from sklearn import metrics
from sklearn.metrics import mean_squared_error, median_absolute_error, mean_squared_error, r2_score, PredictionErrorDisplay
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import scipy.stats as stats
import scipy as sp 
import statsmodels.api
import time
import seaborn as sns
import re
from scipy.stats import ttest_ind
import statsmodels.stats.multitest as smm
import statsmodels.formula.api as smf
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from scipy.stats import f_oneway, chi2_contingency
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import Lasso
from sklearn.model_selection import GroupKFold
import scipy.stats
import statsmodels.api as sm
from scipy.stats import pearsonr

#### Lasso plots

In [ ]:
# Read in meds input previously made
with open(r'avg_coefficients_lasso_1h_df.pkl', 'rb') as handle:
    avg_coefficients_lasso_1h_df = pickle.load(handle)
with open(r'avg_coefficients_lasso_4h_df.pkl', 'rb') as handle:
    avg_coefficients_lasso_4h_df = pickle.load(handle)
with open(r'avg_coefficients_lasso_12h_df.pkl', 'rb') as handle:
    avg_coefficients_lasso_12h_df = pickle.load(handle)

In [ ]:
class_effect_coeff_1h_df=pd.read_excel(r'avg_coefficients_lasso_1h.xlsx')

In [ ]:
class_effect_coeff_4h_df=pd.merge(class_effect_coeff_1h_df,avg_coefficients_lasso_4h_df, on='Feature', how='outer').drop(['Coefficient_1h','Abs_Coefficient_1h'],axis=1)

In [ ]:
class_effect_coeff_12h_df=pd.merge(class_effect_coeff_1h_df,avg_coefficients_lasso_12h_df, on='Feature', how='outer').drop(['Coefficient_1h','Abs_Coefficient_1h'],axis=1)

In [ ]:
class_effect_coeff_12h_df=class_effect_coeff_12h_df.dropna()

In [ ]:
keywords = ["antidiabetic"]

# Use regex to match any of the keywords (case-insensitive)
filtered_df = class_effect_coeff_12h_df[
    class_effect_coeff_12h_df['Class'].str.contains('|'.join(keywords), case=False, na=False)
]
# Sort by 'Value' first (ascending), then sort 'Name' alphabetically
filtered_df_sorted = filtered_df.sort_values(by=['Class', 'Feature'], ascending=[True, True])

In [ ]:
filtered_df_sorted.to_csv('lasso_select_rx.csv')

In [ ]:
filtered_df_sorted["Feature"] = ['glimepiride 1mg PO',
 'glimepiride 2mg PO',
 'glimepiride 4mg PO',
 'glipizide 10mg PO',
 'glipizide 10mg PO ER',
 'glipizide 5mg PO',
 'glipizide 5mg PO ER',
 'glyburide 1.25mg PO',
 'glyburide 5mg PO',
 'insulin aspart SQ',
 'insulin glargine SQ',
 'insulin lispro SQ',
 'insulin NPH & regular 70/30 SQ',
 'insulin NPH SQ',
 'insulin regular SQ',
 'insulin regular IV',
 'metformin 1000mg PO',
 'metformin 500mg PO',
 'nateglinide 120mg PO',
 'nateglinide 60mg PO',
 'pioglitazone 15mg PO',
 'pioglitazone 30mg PO',
 'pioglitazone 45mg PO',
 'sitagliptin 100mg PO',
 'sitagliptin 25mg PO',
 'sitagliptin 50mg PO']

# TODO drop rows with esmolol, labetalol, metop succ
plt.figure(figsize=(12,5))
sns.barplot(data=filtered_df_sorted, x='Feature', y='Average Coefficient', color='blue')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
#plt.xlabel('Drug Class',fontsize=15)
plt.ylabel("Coefficient",fontsize=17)
plt.xlabel("Drug",fontsize=17)
# Adjust layout for clarity
plt.xticks(fontsize=12)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=15) 

#plt.legend(title="Drug Class", title_fontsize=12, fontsize=12, loc="best")  # Adjust size
# Rotate x-axis labels
plt.xticks(rotation=90)  # Adjust the angle as needed
plt.tight_layout()
plt.savefig('figures_revision1/lasso_select_drugs.svg')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.barplot(data=class_effect_coeff_12h_df.sort_values(by='Class'), x='Class', y='Average Coefficient', color='blue')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
#plt.xlabel('Drug Class',fontsize=15)
plt.ylabel("Coefficient",fontsize=17)
plt.xlabel("Drug Class",fontsize=17)
# Adjust layout for clarity
plt.xticks(fontsize=12)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=15) 
# Rotate x-axis labels
plt.xticks(rotation=90)  # Adjust the angle as needed
plt.tight_layout()
plt.savefig('figures_revision1/lasso_class.svg')
plt.show()

In [ ]:
# Count the total rows where average coefficient is exactly 0
total_zero_coeff = class_effect_coeff_12h_df[class_effect_coeff_12h_df["Average Coefficient"] == 0].shape[0]

# Count how many of those rows have BG effect == "yes"
yes_count = class_effect_coeff_12h_df[(class_effect_coeff_12h_df["Average Coefficient"] == 0) & (class_effect_coeff_12h_df["BG effect"] == "yes")].shape[0]

# Calculate the percentage
percentage = (yes_count / total_zero_coeff) * 100 if total_zero_coeff > 0 else 0

print(f"Percentage of rows with average coefficient = 0 that had a BG effect of 'yes': {percentage:.2f}%")

In [ ]:
class_effect_coeff_12h_df['Absolute Coefficient'] = class_effect_coeff_12h_df['Average Coefficient'].abs()

In [ ]:
# Define cutoff values with finer resolution in the lower range
cutoffs = np.concatenate([
    np.linspace(0, 1, 20),  # Finer resolution between 0 and 1
    np.linspace(1.2, class_effect_coeff_12h_df['Absolute Coefficient'].max(), 10)])

# Calculate percentage of "Yes" features at each cutoff
percent_yes = [
    (class_effect_coeff_12h_df[class_effect_coeff_12h_df['Absolute Coefficient'] >= cutoff]['BG effect'] == 'yes').mean() * 100 
    for cutoff in cutoffs]

# Create the plot
plt.figure(figsize=(4.5, 3))
plt.step(cutoffs, percent_yes, where='post', color='b', linewidth=2, label="% Known BG Effect")

# Formatting
plt.xlabel("Absolute Coefficient Threshold", fontsize=12)
plt.ylabel("% Features with Known Effect", fontsize=11)
plt.xticks(np.arange(0, 2.5, 0.2),fontsize=12)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=12) 
plt.xlim(-0.1, 2.3)  # Adjust x-axis limits to include the full range cleanly
plt.ylim(0, 105)  # Keep percentage within reasonable bounds
#plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig('figures_revision1/lasso_perc_effect.svg')
plt.show()

#### Trajectory

In [ ]:
# Read in meds input previously made
with open(r'med_lab_static_output_12h_imputed.pkl', 'rb') as handle:
    med_lab_static_output = pickle.load(handle)

In [ ]:
# Convert boolean columns to integers (vectorized)
med_lab_static_output = med_lab_static_output.convert_dtypes()  # Ensures proper dtypes for conversion
med_lab_static_output.loc[:, med_lab_static_output.dtypes == 'boolean'] = med_lab_static_output.loc[:, med_lab_static_output.dtypes == 'boolean'].astype(int)

In [ ]:
def add_trajectory_features(df):
    """
    Add trajectory features for BG_VALUE within each CSN, including changes, slope, and variability.
    """
    # Remove rows where BG_VALUE > 1000
    df = df[df['BG_VALUE'] <= 1000]
    df = df.sort_values(by=['CSN', 'BG_RESULT_TIME'])
    # Calculate BG change
    df['BG_CHANGE'] = df.groupby('CSN')['BG_VALUE'].diff()
    # Replace NaN values resulting from diff() with 0 (e.g., first row of each group)
    df.fillna(0, inplace=True)
    return df
# Apply to your dataset
df = add_trajectory_features(med_lab_static_output)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print('df shape before drop nan', df.shape)
df = df.dropna()
print('df shape after drop nan', df.shape)

In [ ]:
y = df['BG_CHANGE']
X = df.drop(columns=['BG_VALUE','MRN', 'BG_RESULT_TIME','BG_CHANGE'])

In [ ]:
# Prepare features (X) and target (y)
groups = df['CSN']

# Create a GroupShuffleSplit instance
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Perform the split
for train_idx, test_idx in gss.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Verify no overlap in CSNs
train_csns = set(X_train['CSN'])
test_csns = set(X_test['CSN'])
assert train_csns.isdisjoint(test_csns), "Train and test sets have overlapping CSNs"

# Number of folds for cross-validation
n_splits = 10

# Initialize lists to store coefficients from each fold
all_coefficients_lasso = []

# Define GroupKFold cross-validator
gkf = GroupKFold(n_splits=n_splits)

# Loop over the folds, ensuring no CSN overlap
for train_idx, test_idx in gkf.split(X_train, y_train, groups=X_train['CSN']):
    X_train_fold, X_test_fold = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_train_fold, y_test_fold = y_train.iloc[train_idx], y_train.iloc[test_idx]

    # Fit the Lasso model
    lasso_fold = Lasso(alpha=0.1)
    lasso_fold.fit(X_train_fold, y_train_fold)
    all_coefficients_lasso.append(lasso_fold.coef_)

# Convert list of coefficients to numpy arrays
all_coefficients_lasso = np.array(all_coefficients_lasso)

# Calculate the average coefficients across all folds
avg_coefficients_lasso = np.mean(all_coefficients_lasso, axis=0)

# Final model fitting and evaluation
final_lasso = Lasso(alpha=0.1)
final_lasso.fit(X_train, y_train)
y_pred_lasso = final_lasso.predict(X_test)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mse_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

# Print results
print(f"Final Lasso Model - MSE: {mse_lasso:.4f}, RMSE: {rmse_lasso:.4f}, R2: {r2_lasso:.4f}")

# Store and print the average coefficients (optional)
avg_coefficients_lasso_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Average Coefficient': avg_coefficients_lasso
}).sort_values(by='Average Coefficient', ascending=False)

In [ ]:
class_effect_coeff_12h_df=pd.merge(class_effect_coeff_1h_df,avg_coefficients_lasso_df, on='Feature', how='outer').drop(['Coefficient_1h','Abs_Coefficient_1h'],axis=1)

In [ ]:
plt.figure(figsize=(12,5))
sns.barplot(data=class_effect_coeff_12h_df.sort_values(by='Class'), x='Class', y='Average Coefficient', color='blue')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
#plt.xlabel('Drug Class',fontsize=15)
plt.ylabel("Effect Size",fontsize=17)
plt.xlabel("Drug Class",fontsize=17)
# Adjust layout for clarity
plt.xticks(fontsize=12)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=15) 
# Rotate x-axis labels
plt.xticks(rotation=90)  # Adjust the angle as needed
plt.tight_layout()
#plt.savefig('figures_revision1/lasso_class.svg')
plt.show()

In [ ]:
keywords = ["antidiabetic"]

# Use regex to match any of the keywords (case-insensitive)
filtered_df = class_effect_coeff_12h_df[
    class_effect_coeff_12h_df['Class'].str.contains('|'.join(keywords), case=False, na=False)
]
# Sort by 'Value' first (ascending), then sort 'Name' alphabetically
filtered_df_sorted = filtered_df.sort_values(by=['Class', 'Feature'], ascending=[True, True])

In [ ]:
plt.figure(figsize=(12, 9))
sns.barplot(data=filtered_df_sorted, x='Feature', y='Average Coefficient', hue='Class')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
#plt.xlabel('Drug Class',fontsize=15)
plt.ylabel("Effect Size",fontsize=17)
plt.xlabel("Medication",fontsize=17)
# Adjust layout for clarity
plt.xticks(fontsize=12)  # Set tick marks from 0 to 2.4 every 0.2
plt.yticks(fontsize=15) 

plt.legend(title="Drug Class", title_fontsize=12, fontsize=12, loc="best")  # Adjust size
# Rotate x-axis labels
plt.xticks(rotation=90)  # Adjust the angle as needed
plt.tight_layout()
#plt.savefig('figures_revision1/lasso_select_drugs.svg')
plt.show()

#### Comparison of Lasso and PSM coefficients

In [ ]:
psm_df=pd.read_csv(r'psm_results_df.csv')

In [ ]:
merge=psm_df.merge(class_effect_coeff_12h_df, how='inner', on='Feature')

In [ ]:
# Calculate the difference between Value_lmm and Value_lass
merge["Difference"] = np.abs(merge["coef"] - merge["Average Coefficient"])

In [ ]:
sig_merge=merge[merge['adjusted_p_value']<0.01]

In [ ]:
# Prepare data
x = merge['coef']
y = merge['Average Coefficient']
r, _ = pearsonr(x, y)

# Fit linear regression model
X = sm.add_constant(x)  # Add intercept
model = sm.OLS(y, X).fit()
predictions = model.get_prediction(X)
summary_frame = predictions.summary_frame(alpha=0.05)  # 95% CI

# Create scatter plot
plt.figure(figsize=(6, 5))
plt.scatter(x, y, color='blue', alpha=1, s=50, label='Data')

# Plot regression line (black)
plt.plot(x, summary_frame['mean'], color='black', linewidth=2, label='Best Fit Line')

# Add annotation
plt.text(0.05, 0.95, f"R = {r:.2f}",
         transform=plt.gca().transAxes,
         fontsize=22,
         verticalalignment='top',
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))

# Labels and formatting
plt.xlabel("PSM Coefficient", fontsize=25)
plt.ylabel("Lasso Coefficient", fontsize=25)
plt.xticks(fontsize=22)
plt.yticks(fontsize=22)
plt.tight_layout()
#plt.legend()
plt.savefig("lasso_PSM_with_fit.svg")
plt.show()
